# Raw Docling + Docling-Graph Walkthrough

This notebook drops down to the **raw upstream libraries** —
`docling` and `docling_graph` — and walks a document through the full
extraction arc without our application's wrappers, Celery tasks, or
FastAPI services in the loop.

It mirrors, line-for-line where possible, what our two services
actually do:

| Step in this notebook | What it reproduces from our repo |
|---|---|
| Docling conversion | `docker/docling/app/converter.py` |
| Docling JSON export | `docker/docling/app/converter.py:242` |
| PipelineConfig build | `docker/docling-graph/app/config_builder.py:109-179` |
| OllamaChatClient injection | `docker/docling-graph/app/ollama_clients.py:get_docling_llm_client` |
| Pass template resolution | `docker/docling-graph/app/bundles.py` |
| `run_pipeline(config)` | `docker/docling-graph/app/main.py:428` |
| Provenance build | `docker/docling-graph/app/provenance.py` |

**Why this notebook exists**

- Every step is explicit — no Celery, no HTTP, no MinIO.
- Every config value is inline and annotated — you can see exactly what
  our app passes to `PipelineConfig(**kwargs)`.
- Useful for: debugging extraction regressions, testing schema changes
  without a full pipeline run, evaluating prompt / config tweaks before
  committing to a rebuild.

**OllamaPool refactor (2026-04-28).** The docling-graph service now
bypasses LiteLLM entirely. It builds an `OllamaChatClient` against an
`OllamaPool` (URLs from `OLLAMA_LLM_BASE_URLS`, falling back to the
singular `OLLAMA_LLM_BASE_URL` then `OLLAMA_BASE_URL`) and injects it
into `PipelineConfig(llm_client=...)`. §5 below mirrors that injection
so the notebook traffics through the same code path as production.

**Prerequisites**

1. Main stack up: `docker compose up -d` (for `ollama`, and optionally
   `docling` / `docling-graph` — we don't call those services here).
2. Jupyter sidecar up: `docker compose -f docker-compose.jupyter.yml up -d`.
3. Test file under `notebooks/` on the host; it appears at
   `/app/notebooks/<name>` inside the container.

**Versions present in the Jupyter container at time of writing**

- `docling==2.90.0`
- `docling-graph==1.4.4`
- `docling-core==2.74.0`
- `pydantic==2.12.5`

`litellm` is still installed (transitive dep) but not in our hot path.


## 0. Configuration (edit these)

In [ ]:
from pathlib import Path
import os

# Pick any file under ./notebooks/ on the host.
FILE_PATH = Path("/app/notebooks/SA-2 Surface-to-Air Missile _ National Museum of the United States Air Force™ _ Display.pdf")

# Ollama URL pool. Production reads the JSON-array env var
#   OLLAMA_LLM_BASE_URLS=["http://10.0.1.121:11434","http://10.0.1.122:11434",...]
# (cascade: plural → singular OLLAMA_LLM_BASE_URL → OLLAMA_BASE_URL).
# We use the same cascade here so the notebook's pool matches production.
import json as _json

def _resolve_ollama_urls() -> list[str]:
    raw = os.environ.get("OLLAMA_LLM_BASE_URLS", "").strip()
    if raw:
        try:
            parsed = _json.loads(raw)
            if isinstance(parsed, list) and parsed:
                return [u for u in parsed if isinstance(u, str) and u.strip()]
        except _json.JSONDecodeError:
            pass
    single = os.environ.get("OLLAMA_LLM_BASE_URL", "").strip()
    if single:
        return [single]
    return [os.environ.get("OLLAMA_BASE_URL", "http://ollama:11434")]

OLLAMA_URLS     = _resolve_ollama_urls()
OLLAMA_BASE_URL = OLLAMA_URLS[0]   # for the reachability probe in §1

# LLM model — same default our config_builder uses for extraction (.env:
# DOCLING_GRAPH_LLM_MODEL=gpt-oss:120b at time of writing).
LLM_PROVIDER = os.environ.get("DOCLING_GRAPH_LLM_PROVIDER", "ollama")
LLM_MODEL    = os.environ.get("DOCLING_GRAPH_LLM_MODEL",    "gpt-oss:120b")

# Which pass to run. One of the 12 keys defined in s4-load (5 radar
# field-groups + 6 missile field-groups + system_links).
PASS_NAME = "radar_identity"

assert FILE_PATH.is_file(), f"FILE_PATH not found in container: {FILE_PATH}"
print("FILE_PATH    :", FILE_PATH)
print("OLLAMA URLS  :", OLLAMA_URLS)
print("LLM          :", f"{LLM_PROVIDER}/{LLM_MODEL}")
print("PASS_NAME    :", PASS_NAME)


## 1. Environment + library versions

Verify the three libraries we actually call are importable and that
Ollama is reachable. Our docling service (`docker/docling/app/main.py`)
does its own model-load probe at startup; here we just check imports.


In [ ]:
import importlib.metadata as md
import requests

for pkg in ("docling", "docling-graph", "docling-core", "pydantic"):
    try:
        print(f"  {pkg:16}: {md.version(pkg)}")
    except Exception as exc:
        print(f"  {pkg:16}: NOT INSTALLED ({exc})")

# Confirm the OllamaPool client is importable from app.services. Production
# uses get_llm_client() / get_vlm_client() / get_embedding_client() from
# app.services.ollama_clients; §5 below builds an OllamaChatClient directly
# against the pool to mirror docker/docling-graph/app/ollama_clients.py.
try:
    from app.services.ollama_pool_client import OllamaPool, OllamaChatClient
    print(f"  pool_client     : OK ({OllamaPool.__module__})")
except Exception as exc:
    print(f"  pool_client     : NOT IMPORTABLE ({exc})")

# Ollama reachability — probe the first URL in the pool. (We don't fan out
# here; the OllamaChatClient in §5 will pick URLs via the pool.)
try:
    r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
    r.raise_for_status()
    tags = [m["name"] for m in r.json().get("models", [])]
    print(f"\nOllama reachable: {OLLAMA_BASE_URL}")
    print(f"  models loaded: {tags[:6]}{'…' if len(tags) > 6 else ''}")
    assert any(t.startswith(LLM_MODEL.split(':')[0]) for t in tags), \
        f"Expected a {LLM_MODEL} variant in Ollama. Run: ollama pull {LLM_MODEL}"
except Exception as exc:
    print(f"\nOllama check FAILED: {exc}")
    raise


## 2. Docling conversion (raw library)

Our Docling service wraps `docling.document_converter.DocumentConverter`
with a handful of options tuned for our corpus. We reproduce the same
setup here so the `DoclingDocument` this notebook produces is
**byte-equivalent** to what the service would emit.

**What's reproduced exactly from `docker/docling/app/converter.py`:**

- **PDF pipeline options** (`_build_pdf_pipeline_options`, lines 140-168):
  - `accelerator_options.device="cuda"` — use the GPU.
  - `do_ocr=True` + `EasyOcrOptions(lang=["en"], use_gpu=True)`.
  - `do_table_structure=True` with `TableFormerMode.FAST` + cell matching.
  - `do_formula_enrichment=True`, `do_code_enrichment=True`.
  - `generate_picture_images=True`, `generate_page_images=True`,
    `images_scale=1.0`.
  - `do_picture_description=False` — picture VLM captions are added by
    the app pipeline *post-conversion* via Ollama (not by Docling).

- **`DocumentConverter`** constructed with `PdfFormatOption` for PDF /
  IMAGE and `SimplePipeline` for Word / PowerPoint / Excel / HTML /
  Markdown (converter.py:109-130).

The one thing we skip here: the service's `_patch_pil_crop()`
monkey-patch (converter.py:30-75). That patches PIL to survive
malformed bounding boxes from docling-core; for a single-file run it
rarely matters and only obscures the library's actual behavior.


In [ ]:
import time, io, base64
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions, EasyOcrOptions,
    TableStructureOptions, TableFormerMode,
)
from docling.datamodel.accelerator_options import AcceleratorOptions
from docling.document_converter import (
    DocumentConverter, PdfFormatOption,
    WordFormatOption, PowerpointFormatOption,
    ExcelFormatOption, HTMLFormatOption, MarkdownFormatOption,
)
from docling.pipeline.simple_pipeline import SimplePipeline

pdf_options = PdfPipelineOptions(
    accelerator_options=AcceleratorOptions(device="cuda"),
    do_ocr=True,
    ocr_options=EasyOcrOptions(lang=["en"], use_gpu=True),
    do_table_structure=True,
    table_structure_options=TableStructureOptions(
        do_cell_matching=True,
        mode=TableFormerMode.FAST,
    ),
    do_formula_enrichment=True,
    do_code_enrichment=True,
    generate_picture_images=True,
    generate_page_images=True,
    images_scale=1.0,
    do_picture_description=False,
)

converter = DocumentConverter(
    allowed_formats=[
        InputFormat.PDF, InputFormat.IMAGE,
        InputFormat.DOCX, InputFormat.PPTX, InputFormat.XLSX,
        InputFormat.HTML, InputFormat.MD, InputFormat.ASCIIDOC, InputFormat.CSV,
    ],
    format_options={
        InputFormat.PDF:   PdfFormatOption(pipeline_options=pdf_options),
        InputFormat.IMAGE: PdfFormatOption(pipeline_options=pdf_options),
        InputFormat.DOCX:  WordFormatOption(pipeline_cls=SimplePipeline),
        InputFormat.PPTX:  PowerpointFormatOption(pipeline_cls=SimplePipeline),
        InputFormat.XLSX:  ExcelFormatOption(pipeline_cls=SimplePipeline),
        InputFormat.HTML:  HTMLFormatOption(pipeline_cls=SimplePipeline),
        InputFormat.MD:    MarkdownFormatOption(pipeline_cls=SimplePipeline),
    },
)

t0 = time.monotonic()
result = converter.convert(source=str(FILE_PATH))
elapsed_ms = (time.monotonic() - t0) * 1000

doc = result.document
print(f"Converted in {elapsed_ms:.0f} ms")
print(f"  pages   : {len(doc.pages)}")
print(f"  texts   : {len(doc.texts)}")
print(f"  pictures: {len(doc.pictures)}")
print(f"  tables  : {len(doc.tables)}")


## 3. Inspect the DoclingDocument

The converter returns a `ConversionResult`; `result.document` is the
`DoclingDocument` (docling-core type). Everything downstream — our
chunking, our graph extraction, the Docling-Graph library — consumes
either `doc.export_to_markdown()` or `doc.export_to_dict()` (the JSON
registry form).

Our docling service returns both via its `/convert` endpoint:
- `markdown` — cleaned markdown for text chunking.
- `document_json` — the JSON registry for graph extraction.


In [ ]:
markdown = doc.export_to_markdown()
doc_json = doc.export_to_dict()

print("top-level JSON keys :", list(doc_json.keys()))
print("texts in JSON       :", len(doc_json.get("texts", [])))
print("pictures in JSON    :", len(doc_json.get("pictures", [])))
print("tables in JSON      :", len(doc_json.get("tables", [])))
print("furniture entries   :", len(doc_json.get("furniture", {}).get("children", [])))

print("\n--- Markdown (first 600 chars) ---")
print(markdown[:5000])


In [ ]:
# First non-empty text item from the Docling JSON.
# This is the granularity the library sees; our app's DocumentElement
# table is a curated subset (pipeline.py:2749-2778).
first_text = next((t for t in doc_json.get("texts", []) if t.get("text", "").strip()), None)
if first_text:
    keys = [k for k in first_text.keys() if k != "text"]
    print("first text item keys:", keys)
    print("  self_ref :", first_text.get("self_ref"))
    print("  label    :", first_text.get("label"))
    print("  prov[0]  :", first_text.get("prov", [{}])[0])
    print("  text     :", first_text.get("text")[:2000])


## 4. Pass template — a Pydantic class with catalog conventions

Docling-Graph's delta extractor is driven by a **Pydantic template
class**. Every field description, `Field.examples`, and the `edge()`
helper's metadata flows into the LLM prompt via the library's
`build_delta_node_catalog` + `build_catalog_prompt_block` pipeline.

Our pass templates live under
`ontology_bundles/air_defense_v3/extraction_schemas/<pass>.py`:

- `RadarDomainPass`
- `MissileDomainPass`
- `SystemLinksPass` (relationships-only, takes upstream entities)

**Conventions each class follows** (spec §Template Basics / §4.8):
- Pass roots: `ConfigDict(is_entity=True, graph_id_fields=[])`.
- Primary entities: `ConfigDict(is_entity=True, graph_id_fields=[...])`
  with at least one required identity field. `RadarSystemEntity` and
  `MissileSystemEntity` are **flat** (all checklist fields at the top
  level) — no nested subcomponent classes, no HAS_* edges.
- Pass-root entity-list fields declared via the `edge()` helper
  (label=`CONTAINS`) so GraphConverter walks into them.

Service-side resolution happens in
`docker/docling-graph/app/bundles.py`'s `load_pass_template(...)`; we
just import the class directly here.


In [ ]:
from importlib import import_module
from pydantic import ConfigDict

PASS_MODULES = {
    # ── 5 radar field-group passes (replaced legacy radar_domain) ─────────
    "radar_identity":   ("ontology_bundles.air_defense_v3.extraction_schemas.radar_identity",   "RadarIdentityPass"),
    "radar_power_rf":   ("ontology_bundles.air_defense_v3.extraction_schemas.radar_power_rf",   "RadarPowerRfPass"),
    "radar_antenna":    ("ontology_bundles.air_defense_v3.extraction_schemas.radar_antenna",    "RadarAntennaPass"),
    "radar_timing":     ("ontology_bundles.air_defense_v3.extraction_schemas.radar_timing",     "RadarTimingPass"),
    "radar_modulation": ("ontology_bundles.air_defense_v3.extraction_schemas.radar_modulation", "RadarModulationPass"),
    # ── 6 missile field-group passes (replaced legacy missile_domain) ─────
    "missile_identity":     ("ontology_bundles.air_defense_v3.extraction_schemas.missile_identity",     "MissileIdentityPass"),
    "missile_kinematics":   ("ontology_bundles.air_defense_v3.extraction_schemas.missile_kinematics",   "MissileKinematicsPass"),
    "missile_guidance":     ("ontology_bundles.air_defense_v3.extraction_schemas.missile_guidance",     "MissileGuidancePass"),
    "missile_airframe":     ("ontology_bundles.air_defense_v3.extraction_schemas.missile_airframe",     "MissileAirframePass"),
    "missile_speed_timing": ("ontology_bundles.air_defense_v3.extraction_schemas.missile_speed_timing", "MissileSpeedTimingPass"),
    "missile_propulsion":   ("ontology_bundles.air_defense_v3.extraction_schemas.missile_propulsion",   "MissilePropulsionPass"),
    # ── cross-domain relationship pass ─────────────────────────────────────
    "system_links":     ("ontology_bundles.air_defense_v3.extraction_schemas.system_links",     "SystemLinksPass"),
}
mod_path, cls_name = PASS_MODULES[PASS_NAME]
template_cls = getattr(import_module(mod_path), cls_name)

cfg = dict(template_cls.model_config) if isinstance(template_cls.model_config, dict) else {}
print(f"Pass template : {cls_name}")
print(f"  module      : {mod_path}")
print(f"  is_entity   : {cfg.get('is_entity')}")
print(f"  graph_id_fields : {cfg.get('graph_id_fields')}")
print()
print(f"Top-level fields ({len(template_cls.model_fields)}):")
for fname, finfo in template_cls.model_fields.items():
    extra = getattr(finfo, "json_schema_extra", None) or {}
    edge_label = extra.get("edge_label") if isinstance(extra, dict) else None
    label = f"  - {fname}"
    if edge_label:
        label += f"  [edge={edge_label}]"
    desc = (finfo.description or "")[:80]
    print(f"{label}: {desc}")


## 5. Build a `PipelineConfig` — exactly how our service does

This is the contract between us and the library. Every knob below
corresponds one-to-one with `DoclingGraphSettings` in
`docker/docling-graph/app/config_builder.py:17-97`, and the kwargs dict
is the literal one our `build_pipeline_config()` assembles
(lines 128-174).

Each field is annotated with:
- **env var:** the name our app resolves from the process environment.
- **default:** our committed default (from the Python class, not from
  the library's default).
- **why:** what it controls at extraction time.

Per-pass override: `delta_quality_min_instances` drops to `1` for
`system_links` (relationships-only pass — producing zero ontology nodes
is legitimate there). We replicate that branch explicitly.


In [ ]:
from typing import Any
from docling_graph import PipelineConfig

# Per-pass overrides from config_builder.py:104-106. system_links is the
# only exception today.
_QUALITY_MIN_INSTANCES_PER_PASS = {"system_links": 1}


def _build_local_llm_client():
    """Mirror docker/docling-graph/app/ollama_clients.py:get_docling_llm_client.

    Production wires a process-cached OllamaChatClient with:
      - schema_transform = sanitize_schema_for_llm  (only available
        inside the docling-graph container; we leave it None here)
      - parse_json_fn    = parse_llm_json_loose     (from app.services.llm_json)
      - client_error_cls = docling_graph.exceptions.ClientError
      - force_json_mode  = service_settings.force_json_mode (default True)

    Caveat: this notebook runs in the Jupyter sidecar where the
    container-local app.prompt_rules is not on PYTHONPATH, so we leave
    schema_transform=None. Output should still match production when
    force_json_mode=True (the schema is replaced with format="json"
    before the post anyway). When force_json_mode=False the lack of
    schema_transform may surface schema-strip discrepancies; rebuild
    inside the docling-graph container if you need exact parity.
    """
    from app.services.ollama_pool_client import OllamaChatClient, OllamaPool
    from app.services.llm_json import parse_llm_json_loose
    from docling_graph.exceptions import ClientError

    pool = OllamaPool(urls=OLLAMA_URLS)
    return OllamaChatClient(
        pool=pool,
        model=LLM_MODEL,
        timeout_s=float(os.environ.get("DOCLING_GRAPH_LLM_TIMEOUT", "7200")),
        temperature=float(os.environ.get("DOCLING_GRAPH_LLM_TEMPERATURE", "0.1")),
        max_tokens=int(os.environ.get("DOCLING_GRAPH_LLM_MAX_TOKENS", "32000")),
        think=os.environ.get("DOCLING_GRAPH_LLM_THINK", "") or None,
        schema_transform=None,                       # see caveat above
        force_json_mode=os.environ.get("DOCLING_GRAPH_FORCE_JSON_MODE", "true").lower() == "true",
        client_error_cls=ClientError,
        parse_json_fn=parse_llm_json_loose,
    )


def build_pipeline_config_local(source: str, template_class, pass_name: str | None = None) -> PipelineConfig:
    """Replica of docker/docling-graph/app/config_builder.py:build_pipeline_config.

    Two differences vs the in-container builder:
      - reads env vars directly (no pydantic_settings dance), so you can
        tweak a value in the notebook and reconfigure instantly;
      - injects an OllamaChatClient (via _build_local_llm_client above)
        rather than letting the library fall back to its own LiteLLMClient.
    """
    env = os.environ.get

    quality_min_instances = int(env("DOCLING_GRAPH_QUALITY_MIN_INSTANCES", "3"))
    if pass_name in _QUALITY_MIN_INSTANCES_PER_PASS:
        quality_min_instances = _QUALITY_MIN_INSTANCES_PER_PASS[pass_name]

    kwargs: dict[str, Any] = {
        # Source file — the library loads this into a DoclingDocument if
        # it isn't already one; we pass the raw PDF path.
        "source": source,

        # --- LLM backend (config_builder.py:96-97, 32-33) -----------------
        "backend":           env("DOCLING_GRAPH_BACKEND",              "llm"),
        "inference":         "local",
        # provider_override / model_override are now inert when we inject an
        # llm_client (pipeline/stages.py short-circuits on llm_client). We
        # keep model_override for log-message parity with production.
        "provider_override": env("DOCLING_GRAPH_LLM_PROVIDER",         "ollama"),
        "model_override":    env("DOCLING_GRAPH_LLM_MODEL",            "gpt-oss:120b"),

        # --- Extraction contract (config_builder.py:34-35) ----------------
        # "delta" = chunk-level emit-nodes+relationships; what the service
        # ships. Alternatives: "direct", "staged".
        "extraction_contract": env("DOCLING_GRAPH_EXTRACTION_CONTRACT", "delta"),
        "processing_mode":     env("DOCLING_GRAPH_PROCESSING_MODE",     "many-to-one"),

        # --- Chunking (config_builder.py:38-45) ---------------------------
        "use_chunking":            env("DOCLING_GRAPH_USE_CHUNKING",          "true").lower() == "true",
        "chunk_max_tokens":    int(env("DOCLING_GRAPH_CHUNK_MAX_TOKENS",      "512")),
        "llm_batch_token_size":int(env("DOCLING_GRAPH_LLM_BATCH_TOKEN_SIZE",  "1024")),
        "parallel_workers":    int(env("DOCLING_GRAPH_PARALLEL_WORKERS",      "5")),
        "staged_pass_retries": int(env("DOCLING_GRAPH_BATCH_SPLIT_MAX_RETRIES","1")),

        # --- Delta resolvers (config_builder.py:48-51) --------------------
        "delta_resolvers_enabled":    env("DOCLING_GRAPH_RESOLVERS_ENABLED", "true").lower() == "true",
        "delta_resolvers_mode":       env("DOCLING_GRAPH_RESOLVERS_MODE",    "semantic"),
        "delta_resolver_fuzzy_threshold":    float(env("DOCLING_GRAPH_RESOLVER_FUZZY_THRESHOLD",    "0.8")),
        "delta_resolver_semantic_threshold": float(env("DOCLING_GRAPH_RESOLVER_SEMANTIC_THRESHOLD", "0.8")),

        # --- Delta quality gate (config_builder.py:54-57) ------------------
        # Upstream lib default is 20 — we lowered to 3, then (in .env) to 1
        # while debugging sparse prose extraction.
        "delta_quality_require_root":       env("DOCLING_GRAPH_QUALITY_REQUIRE_ROOT",   "true").lower() == "true",
        "delta_quality_min_instances":      quality_min_instances,
        "delta_quality_max_parent_lookup_miss": int(env("DOCLING_GRAPH_QUALITY_MAX_PARENT_MISS", "4")),
        "delta_quality_adaptive_parent_lookup":  env("DOCLING_GRAPH_QUALITY_ADAPTIVE_PARENT", "true").lower() == "true",

        # --- Delta normalizer (config_builder.py:60-63) -------------------
        "delta_normalizer_validate_paths":          env("DOCLING_GRAPH_NORMALIZER_VALIDATE_PATHS",       "true").lower() == "true",
        "delta_normalizer_canonicalize_ids":        env("DOCLING_GRAPH_NORMALIZER_CANONICALIZE_IDS",     "true").lower() == "true",
        "delta_normalizer_strip_nested_properties": env("DOCLING_GRAPH_NORMALIZER_STRIP_NESTED",         "true").lower() == "true",
        "delta_normalizer_attach_provenance":       env("DOCLING_GRAPH_NORMALIZER_ATTACH_PROVENANCE",    "true").lower() == "true",

        # --- Identity filter (post-extraction section-title pruner) -------
        # This is the "docs-recommended safety net" that replaces our old
        # prompt_overrides.py rewrites (removed).
        "delta_identity_filter_enabled": env("DOCLING_GRAPH_IDENTITY_FILTER_ENABLED", "true").lower() == "true",
        "delta_identity_filter_strict":  env("DOCLING_GRAPH_IDENTITY_FILTER_STRICT",  "false").lower() == "true",

        # --- Gleaning (config_builder.py:75-76) ---------------------------
        # 1 extra pass after the primary: "what did you miss?" for recall.
        "gleaning_enabled":        env("DOCLING_GRAPH_GLEANING_ENABLED",        "true").lower() == "true",
        "gleaning_max_passes": int(env("DOCLING_GRAPH_GLEANING_MAX_PASSES",     "2")),

        # --- Structured output (config_builder.py:79-80) ------------------
        "structured_output":        env("DOCLING_GRAPH_STRUCTURED_OUTPUT",       "true").lower() == "true",
        "structured_sparse_check":  env("DOCLING_GRAPH_STRUCTURED_SPARSE_CHECK", "true").lower() == "true",

        # --- LLM overrides (config_builder.py:230-240) --------------------
        # When llm_client is injected (below), connection.base_url is unused
        # and the LiteLLM-specific context_limit / max_output_tokens overrides
        # are no-ops too (they were workarounds for LiteLLM's
        # resolve_effective_model_config, which our path doesn't call).
        # We keep generation/reliability for the library's own retry/limit
        # bookkeeping.
        "llm_overrides": {
            "generation": {
                "temperature": float(env("DOCLING_GRAPH_LLM_TEMPERATURE", "0.1")),
                "max_tokens":   int(env("DOCLING_GRAPH_LLM_MAX_TOKENS", "32000")),
            },
            "reliability": {
                "timeout_s": int(env("DOCLING_GRAPH_LLM_TIMEOUT", "72000")),
            },
        },

        # --- Misc ---------------------------------------------------------
        "dump_to_disk": False,
    }
    if template_class is not None:
        kwargs["template"] = template_class

    # Inject our own OllamaChatClient — mirrors production
    # (config_builder.py:244-245). pipeline/stages.py:470 short-circuits on
    # llm_client, so the library's LiteLLMClient is never instantiated.
    kwargs["llm_client"] = _build_local_llm_client()

    return PipelineConfig(**kwargs)


config = build_pipeline_config_local(
    source=str(FILE_PATH),
    template_class=template_cls,
    pass_name=PASS_NAME,
)

# Show the subset of fields a reader cares about.
print(f"PipelineConfig(")
for k in ("backend", "provider_override", "model_override",
          "extraction_contract", "processing_mode",
          "use_chunking", "chunk_max_tokens", "llm_batch_token_size",
          "delta_quality_min_instances", "delta_identity_filter_enabled",
          "gleaning_enabled", "gleaning_max_passes",
          "structured_output"):
    v = getattr(config, k, "<missing>")
    print(f"  {k:32} = {v!r}")
print(f"  llm_overrides.generation.temperature = {config.llm_overrides.generation.temperature}")
print(f"  llm_client                 = {type(config.llm_client).__name__} "
      f"(model={config.llm_client.model}, urls={config.llm_client.pool.urls})")
print(f")")


## 6. Catalog + prompt (what the LLM actually sees)

We reconstruct the exact system/user prompt the library will send for a
given batch — using production's own chunker and batcher. Four
ingredients go into that prompt:

1. **Chunks** from `DocumentChunker` (HybridChunker + sentence-transformers
   tokenizer, capped at `chunk_max_tokens=512`). This merges Docling's
   raw `texts[]` fragments into semantically coherent chunks, so you do
   *not* see icons / page numbers / single-line UI crumbs as standalone
   chunks. That raw registry is a preview pitfall.
2. **Batches** from `chunk_batches_by_token_limit` packing those chunks
   until each batch totals ≤ `llm_batch_token_size=1024` tokens.
3. **Path catalog** from `build_delta_node_catalog` +
   `build_catalog_prompt_block` — flattened from the template class.
4. **Semantic guide** from `build_delta_semantic_guide` — derived from
   the Pydantic JSON schema.

Then `get_delta_batch_prompt(...)` assembles them with one batch's
content. Adjust `BATCH_INDEX` in the cell below to view different
batches — this is what the LLM gets for each `/extract-pass` call that
production fires for this document.


In [ ]:
from docling_graph.core.extractors.document_chunker import DocumentChunker
from docling_graph.core.extractors.contracts.delta.helpers import chunk_batches_by_token_limit
from docling_graph.core.extractors.contracts.delta.catalog import build_delta_node_catalog
from docling_graph.core.extractors.contracts.delta.schema_mapper import (
    build_catalog_prompt_block, build_delta_semantic_guide,
)
from docling_graph.core.extractors.contracts.delta.prompts import (
    get_delta_batch_prompt, format_batch_markdown,
)
# Match the service-side rewrite: replace the library's system prompt with the
# mention-level-friendly version. Same source of truth both sides import from.
from ontology_bundles._shared.prompt_rules import DELTA_SYSTEM_PROMPT
_original_prompt = get_delta_batch_prompt
def get_delta_batch_prompt(**kw):
    r = _original_prompt(**kw)
    if isinstance(r, dict) and "system" in r:
        r["system"] = DELTA_SYSTEM_PROMPT
    return r

BATCH_INDEX          = 0     # which real batch to display (0 = first)
CHUNK_MAX_TOKENS     = 512   # matches DoclingGraphSettings.docling_graph_chunk_max_tokens
LLM_BATCH_TOKEN_SIZE = 1024  # matches DoclingGraphSettings.docling_graph_llm_batch_token_size

# ── 1. Run production's chunker over this document's DoclingDocument ─
# `doc` is the DoclingDocument produced by Section 2's DocumentConverter call.
chunker = DocumentChunker(
    tokenizer_name="sentence-transformers/all-MiniLM-L6-v2",
    chunk_max_tokens=CHUNK_MAX_TOKENS,
    merge_peers=True,
)
chunks       = chunker.chunk_document(doc)
token_counts = [chunker.tokenizer.count_tokens(c) for c in chunks]
batch_plan   = chunk_batches_by_token_limit(
    chunks, token_counts, max_batch_tokens=LLM_BATCH_TOKEN_SIZE,
)

print(f"chunker produced {len(chunks)} chunks (total {sum(token_counts)} tokens)")
print(f"batched into     {len(batch_plan)} batch(es)")
for i, b in enumerate(batch_plan):
    tokens = sum(t for _, _, t in b)
    print(f"  batch {i:2}: {len(b):2} chunks, {tokens:4} tokens")

if BATCH_INDEX >= len(batch_plan):
    raise SystemExit(
        f"BATCH_INDEX={BATCH_INDEX} out of range (only {len(batch_plan)} batches)."
    )

selected_batch = batch_plan[BATCH_INDEX]
batch_markdown = format_batch_markdown([text for _i, text, _t in selected_batch])

# ── 2. Build catalog + semantic guide from the template ────────────
catalog        = build_delta_node_catalog(template_cls)
catalog_block  = build_catalog_prompt_block(catalog)
schema_dict    = template_cls.model_json_schema()
semantic_guide = build_delta_semantic_guide(template_cls, schema_dict)

first_chunk = chunks[0].strip() if chunks else ""
global_context = (
    first_chunk[:600] + ("..." if len(first_chunk) > 600 else "")
) if first_chunk else None

# ── 3. Assemble the prompt ─────────────────────────────────────────
prompt = get_delta_batch_prompt(
    batch_markdown=batch_markdown,
    schema_semantic_guide=semantic_guide,
    path_catalog_block=catalog_block,
    batch_index=BATCH_INDEX,
    total_batches=len(batch_plan),
    global_context=global_context,
    already_found=None,
)

print(f"\ncatalog paths : {len(catalog.nodes)}")
print(f"catalog_block : {len(catalog_block)} chars")
print(f"semantic_guide: {len(semantic_guide)} chars")
print(f"prompt.system : {len(prompt['system'])} chars")
print(f"prompt.user   : {len(prompt['user'])} chars")
print()
print(f"=== SYSTEM (full) ===\n{prompt['system']}")
print(f"\n=== USER (batch {BATCH_INDEX+1}/{len(batch_plan)}, {sum(t for _,_,t in selected_batch)} tokens) ===")
print(prompt["user"])


## 7. Run one pass via `run_pipeline(config)`

`run_pipeline` is the library's top-level entry point for an
end-to-end extraction against a single source. Our service uses it via
`run_extraction_pass(...)` at
`docker/docling-graph/app/main.py:376-445`.

What happens inside:

1. Parse / normalize the source into a `DoclingDocument`.
2. Chunk the document (if `use_chunking=True`).
3. For each batch: build the prompt from §6 and call the LLM via the
   injected `OllamaChatClient` (which talks `/v1/chat/completions` to
   the URL picked by `OllamaPool.acquire()`), validating against
   `DeltaGraph` JSON schema. Pre-OllamaPool refactor this routed through
   `LiteLLMClient`; the on-the-wire shape is the same but library logs
   in `diagnostics.library_log` now read `Initialized LlmBackend with
   client: OllamaChatClient`.
4. Resolve + normalize + identity-filter the merged graph.
5. Run the quality gate (`delta_quality_*` knobs).
6. Optionally: gleaning pass for recall.
7. Project the graph into the template (`template_instance`) and
   return a `PipelineContext`.

The service then promotes `extracted_models[0]` to `template_instance`
(main.py:435-437) — mirror that here.

> **Tip.** First run on Ollama is slow (model warm-up + constrained-
> decoding compilation). Subsequent runs are much faster.


In [ ]:
from docling_graph import run_pipeline

context = run_pipeline(config)

# Mirror our service's promotion step so downstream code sees the
# populated pass root (main.py:435-437).
extracted = getattr(context, "extracted_models", None)
if isinstance(extracted, list) and extracted:
    context.template_instance = extracted[0]

graph = context.knowledge_graph
meta  = context.graph_metadata
print(f"nodes : {graph.number_of_nodes()}")
print(f"edges : {graph.number_of_edges()}")
print(f"node_types (meta) : {getattr(meta, 'node_types', None)}")
print(f"edge_types (meta) : {getattr(meta, 'edge_types', None)}")
print(f"template_instance : {type(context.template_instance).__name__ if context.template_instance else None}")


In [ ]:
# Inspect what the pass produced — the same shape our service returns in
# ExtractPassResponse.pass_output (main.py:582-590). Three complementary
# views: (1) per-entity identity + populated fields, (2) full JSON dump of
# the template instance, (3) knowledge-graph nodes/edges from the extractor
# context. Flat schemas have no typed edges, so edge counts are expected to
# be 0 or 1 (pass-root → system entities via CONTAINS).
import json as _json_mod
from pprint import pprint

pass_output = (
    context.template_instance.model_dump(mode="json", exclude_none=False)
    if context.template_instance is not None
    else None
)

if pass_output is None:
    print("No template_instance produced — empty extraction or quality-gate failure.")
    print("See diagnostics in the next cell (or the docling-graph debug dir).")
else:
    print(f"pass_output top-level keys: {list(pass_output.keys())}\n")

    # (1) Per-entity quick summary — skip None fields so populated data is visible.
    for key, value in pass_output.items():
        if not isinstance(value, list):
            print(f"  {key}: {type(value).__name__} = {value!r}")
            continue
        print(f"  {key}: {len(value)} entity instance(s)")
        for i, item in enumerate(value):
            if not isinstance(item, dict):
                print(f"    [{i}] {item!r}")
                continue
            populated = {k: v for k, v in item.items() if v is not None and v != []}
            print(f"    [{i}] populated fields ({len(populated)}/{len(item)}):")
            for k, v in populated.items():
                s = repr(v)
                if len(s) > 160:
                    s = s[:157] + "..."
                print(f"         {k}: {s}")
        print()

    # (2) Full JSON dump — no truncation, so you can copy-paste or diff.
    print("=" * 72)
    print("Full pass_output (untrimmed):")
    print("=" * 72)
    print(_json_mod.dumps(pass_output, indent=2, ensure_ascii=False, default=str))

# (3) Knowledge graph — independent view. Library builds this from the
# template instance via GraphConverter; nodes are entities, edges reflect
# edge() fields on the Pydantic classes.
print("\n" + "=" * 72)
print("context.knowledge_graph (networkx DiGraph):")
print("=" * 72)
g = context.knowledge_graph
print(f"nodes: {g.number_of_nodes()}   edges: {g.number_of_edges()}")
for n, data in list(g.nodes(data=True))[:20]:
    nt = data.get("node_type") or data.get("type")
    lbl = data.get("label") or data.get("name")
    print(f"  node[{n}] type={nt!r} label={lbl!r}")
for u, v, data in list(g.edges(data=True))[:20]:
    print(f"  edge {u} -> {v}  {data.get('label', data)}")


## 8. Multi-pass orchestration (radar → missile → system_links)

Our bundle runs three passes in declared order. The first two are
`input_mode=document_only` and run independently; the third,
`system_links`, is `input_mode=document_plus_entity_refs` — it consumes
entities from the first two and emits cross-domain relationships.

The service threads upstream entities through the "Path B preamble"
mechanism (`main.py:387-415`): a formatted block of upstream entity
identities is inserted into `doc_json.texts` + `body.children` so the
LLM sees them in the extraction batch.

Below we run a simplified version: two independent passes,
concatenate their emitted entities as prose, and pass that text as a
batch for the relationships-only pass.

> This is a fidelity shortcut, not a perfect reproduction. The service
> reconstructs the `docling_document_json` shape per pass; here we only
> want to show the sequencing + upstream-ref propagation. For a full
> reproduction of Path B, see `main.py:387-415`.


In [ ]:
def run_pass(pass_name: str):
    mod_path, cls_name = PASS_MODULES[pass_name]
    cls  = getattr(import_module(mod_path), cls_name)
    cfg  = build_pipeline_config_local(str(FILE_PATH), cls, pass_name=pass_name)
    ctx  = run_pipeline(cfg)
    extr = getattr(ctx, "extracted_models", None)
    if isinstance(extr, list) and extr:
        ctx.template_instance = extr[0]
    return ctx


# All 12 active passes from manifest.yaml. We run radar then missile then
# system_links so system_links has upstream entity refs available.
RADAR_PASSES   = ["radar_identity", "radar_power_rf", "radar_antenna",
                  "radar_timing", "radar_modulation"]
MISSILE_PASSES = ["missile_identity", "missile_kinematics", "missile_guidance",
                  "missile_airframe", "missile_speed_timing", "missile_propulsion"]

# WARNING: running all 12 sequentially against gpt-oss:120b on a real PDF can
# take 1-3 hours per pass on a parameter-heavy doc. Pick a subset for a
# quick sanity-check, or run them all and walk away.
PASSES_TO_RUN = ["radar_identity", "missile_identity"]

contexts = {}
has_any_entities = False
for p in PASSES_TO_RUN:
    ctx = run_pass(p)
    contexts[p] = ctx
    nodes = ctx.knowledge_graph.number_of_nodes()
    edges = ctx.knowledge_graph.number_of_edges()
    print(f"{p:<22}: nodes={nodes} edges={edges}")
    if ctx.template_instance is not None:
        has_any_entities = True

# system_links is relationships-only and needs upstream entity refs.
# In the service this happens via Path B preamble injection (main.py:387-415).
# For the notebook we just skip if neither identity pass produced anything.
if has_any_entities and "system_links" in PASS_MODULES:
    ctx_links = run_pass("system_links")
    contexts["system_links"] = ctx_links
    print(f"system_links          : nodes={ctx_links.knowledge_graph.number_of_nodes()} "
          f"edges={ctx_links.knowledge_graph.number_of_edges()}")
else:
    print("system_links          : SKIPPED (no upstream entity refs to link)")


## 9. Build provenance rows (what the service returns per-node)

Our service builds per-node `ExtractionProvenance` rows via
`build_provenance_from_context` (`docker/docling-graph/app/provenance.py`).
These are attached to the `/extract-pass` response and used downstream
to (a) map every extracted entity back to its originating Docling
element and (b) populate `HAS_PROVENANCE` edges in ArcadeDB.

The raw library doesn't build this structure for us — the service
does. Here we replicate it inline so you can see the shape.


In [ ]:
# ExtractionProvenance is defined by docling-graph services' schema. For
# the notebook we'll dump what the library itself attached to each node.
nodes = []
for node_key, data in context.knowledge_graph.nodes(data=True):
    prov = data.get("_provenance") or data.get("provenance")
    if prov:
        nodes.append({"node": node_key, "type": data.get("node_type"), "prov": prov})

print(f"nodes with library-level _provenance: {len(nodes)}")
for n in nodes[:3]:
    print(n)


## 10. What's missing vs. the full pipeline

This notebook covers the **extraction** arc end-to-end using the raw
libraries. The following are deliberately out of scope; they're app-
level concerns our full pipeline (`app/workers/pipeline.py`) handles:

- **MinIO persistence** — raw files, Docling JSON, image artifacts.
- **PostgreSQL persistence** — `document_elements`, `artifacts`,
  `text_chunks`, `image_chunks`, `graph_extractions` rows.
- **Text / image embeddings** — BGE-M3 for text (via Ollama),
  open_clip ViT-L-16-SigLIP2-256 for images.
- **Merge across passes** — `app/services/extraction_merge.py`
  deduplicates `LogicalIdentity`s emitted by different passes and
  validates edges against the ontology validation matrix.
- **ArcadeDB upserts** — entity vertices, chunk vertices,
  `EXTRACTED_FROM`, `HAS_PROVENANCE`, `SAME_PAGE`, etc.
- **Pipeline-run bookkeeping** — `pipeline_runs`, `stage_runs`,
  status transitions.
- **Anchor walker** (`derive_document_anchors`) — deterministic
  structural entities emitted from the Docling document shape with
  **no LLM involvement**: `SECTION`, `FIGURE`, `TABLE`, `IMAGE`
  (uncaptioned pictures), and `TEXT_BLOCK` (body text adjacent to
  figures/images). Edges: `HAS_SECTION`, `HAS_FIGURE`, `HAS_TABLE`,
  `HAS_IMAGE`, `NEAR_TEXT` (figure/image → text block), `CHILD_OF`
  (section nesting). Source: `app/services/docling_anchors.py`;
  called via `derive_document_anchors` task (pipeline.py:4271).

For those, run the `ingest_walkthrough.ipynb` notebook instead — it
uses the real Celery tasks via `.apply(...)` and produces all the side
effects. See §8 in that notebook for anchor walker output.
